# News vector RAG — interactive tour

The deep **`AgensgraphVector`** showcase: real news (CC-News) in a pgvector HNSW
store with a fulltext keyword index, queried five ways — ending in a LangChain
**LCEL RAG chain**.

**Prerequisite — ingest first:**

```bash
cd langchain
NEWS_LIMIT=10000 .venv/bin/python examples/demos/03_news_vector_rag/ingest.py
```

In [1]:
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../03_news_vector_rag
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for _common
sys.path.insert(0, str(HERE))              # this dir, for `rag`

import pandas as pd
import rag                                  # store construction + helpers live here
from _common import agens, config
from _common.models import get_llm
from langchain_agensgraph.vectorstores.agensgraph_vector import SearchType, HybridSearchConfig

vector = rag._vec(SearchType.VECTOR)                 # vector + filtered search
hybrid = rag._vec(SearchType.HYBRID, keyword="keyword")  # vector + keyword (RRF)
graph = agens.make_graph("news", create=False, refresh_schema=False)

def hits_df(hits):
    return pd.DataFrame([
        {"score": round(s, 3), "title": d.metadata.get("title", "")[:55],
         "domain": d.metadata.get("domain", ""), "date": d.metadata.get("date", "")}
        for d, s in hits
    ])

print("connected to", config.url().split("@")[-1])

connected to localhost:55432/agensgraph_demos


## The corpus

In [2]:
n = graph.query('MATCH (n:"Article") RETURN count(n) AS c')[0]["c"]
rng = graph.query('MATCH (n:"Article") WHERE n.date IS NOT NULL RETURN min(n.date) AS lo, max(n.date) AS hi')[0]
print(f"{n:,} chunks · dates {rng['lo']} .. {rng['hi']}")
pd.DataFrame(graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS chunks ORDER BY chunks DESC LIMIT 8'
))

89,445 chunks · dates 2017-01-01 .. 2018-07-15


,domain,chunks
0,nationalpost.com,9660
1,www.taiwannews.com.tw,8275
2,abcnews.go.com,6812
3,www.nigeriatoday.ng,4917
4,www.wave3.com,3698
5,www.yahoo.com,3243
6,www.nytimes.com,3189
7,www.ocregister.com,2197


## (a) Vector semantic search (HNSW)

In [3]:
hits_df(vector.similarity_search_with_score("artificial intelligence and machine learning", k=5))

,score,title,domain,date
0,0.633,Top 5: Things AI might actually be good for,www.techrepublic.com,2018-02-02
1,0.622,Teaching Self-Learning Machines to Forget,www.industryweek.com,2017-12-11
2,0.614,Why some of the world's biggest companies are ...,www.techrepublic.com,2017-12-11
3,0.602,Teaching Self-Learning Machines to Forget,www.industryweek.com,2017-12-11
4,0.575,A Team of MIT Scientists Taught an AI to Get E...,www.yahoo.com,2017-12-11


## (b) Metadata-filtered search

MongoDB-style filter operators on the chunk metadata.

In [4]:
domains = [r["domain"] for r in graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS c ORDER BY c DESC LIMIT 3')]
# A date the record did not carry is absent rather than "", so IS NOT NULL is the whole
# test — and the index answers it from its first entry.
lo = graph.query(
    'MATCH (n:"Article") WHERE n.date IS NOT NULL RETURN min(n.date) AS lo')[0]["lo"]
flt = {"$and": [{"domain": {"$in": domains}}, {"date": {"$gte": lo}}]}
print("filter =", flt)
hits_df(vector.similarity_search_with_score("business and technology", k=5, filter=flt))


filter = {'$and': [{'domain': {'$in': ['nationalpost.com', 'www.taiwannews.com.tw', 'abcnews.go.com']}}, {'date': {'$gte': '2017-01-01'}}]}


,score,title,domain,date
0,0.510,Avnet Journal Reveals Keys to Hardening the Io...,www.taiwannews.com.tw,2018-04-24
1,0.501,ABB and the Economist Launch Automation Readin...,www.taiwannews.com.tw,2018-04-23
2,0.470,Top Insights on the Cloud Migration Services M...,www.taiwannews.com.tw,2018-05-30
3,0.459,Top Factors Driving the Global Smart Ceiling F...,www.taiwannews.com.tw,2018-04-25
4,0.459,Top Factors Driving the Global Motorcycle Head...,www.taiwannews.com.tw,2018-05-31


## (c) Hybrid search (vector + keyword, RRF fusion)

`HybridSearchConfig` tunes the reciprocal-rank-fusion weighting.

In [5]:
hits_df(hybrid.similarity_search_with_score(
    "stock market", k=5, hybrid_config=HybridSearchConfig(keyword_weight=2.0)))

,score,title,domain,date
0,0.049,Investment pros staying calm after rate fears ...,www.wafb.com,2018-02-03
1,0.032,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
2,0.032,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
3,0.031,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
4,0.031,Investment pros staying calm after rate fears ...,abcnews.go.com,2018-02-03


## (d) `search_options` — pgvector's recall control

`hnsw.ef_search` is how many candidates the index considers before ranking them, so
raising it trades latency for recall. Over-fetching `k` does not: a larger `LIMIT`
returns the same rows, because the index has already stopped looking.
`effective_search_ratio` is still accepted and does nothing but over-fetch.


In [6]:
for ef in (20, 200):
    print(f"hnsw.ef_search = {ef}")
    display(hits_df(vector.similarity_search_with_score(
        "sports", k=5, filter={"date": {"$gte": lo}},
        search_options={"hnsw.ef_search": ef})))


hnsw.ef_search = 20


,score,title,domain,date
0,0.445,"Bluiett scores 25 points, No. 13 Xavier beats ...",nationalpost.com,2017-12-10
1,0.399,Docking: A look back at 2016's top sports moments,www.chch.com,2017-01-17
2,0.376,UCF kicker's YouTube profits may be violation ...,www.wave3.com,2017-06-14
3,0.367,The Latest: Play ended for the day at French O...,www.wave3.com,2018-05-30
4,0.363,Trump’s Blood Sport Politics,www.nytimes.com,2018-02-02


hnsw.ef_search = 200


,score,title,domain,date
0,0.445,"Bluiett scores 25 points, No. 13 Xavier beats ...",nationalpost.com,2017-12-10
1,0.399,Docking: A look back at 2016's top sports moments,www.chch.com,2017-01-17
2,0.376,UCF kicker's YouTube profits may be violation ...,www.wave3.com,2017-06-14
3,0.367,The Latest: Play ended for the day at French O...,www.wave3.com,2018-05-30
4,0.363,Trump’s Blood Sport Politics,www.nytimes.com,2018-02-02


## (e) RAG — `as_retriever()` in an LCEL chain

The vector store becomes a LangChain retriever, composed with a prompt + LLM
into a cited, grounded answer.

In [7]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vector.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('domain','?')} {d.metadata.get('date','')}] "
        f"{d.metadata.get('title','')}\n{d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the news snippets below. Cite the source "
               "domains you rely on. If they don't cover it, say so."),
    ("human", "Question: {question}\n\nNews snippets:\n{context}"),
])
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | get_llm() | StrOutputParser()
)

print(chain.invoke("How is artificial intelligence being used in business?"))

Artificial intelligence (AI) is being utilized in various ways within businesses, primarily in the following areas:

1. **Human Resources and Hiring**: AI is increasingly being used to manage human employees, particularly in hiring processes. Companies are leveraging AI to sort through resumes, rank candidates, and analyze their responses, as seen with Unilever's use of AI to enhance their hiring efficiency (TechRepublic).

2. **Workforce Management**: AI tools are helping businesses make more objective decisions based on data, which can improve management practices. This includes the potential for AI to assist in identifying employee potential and optimizing workforce configurations (TechRepublic).

3. **Customer Service**: AI is enhancing customer service by providing support to human agents, processing natural language, and improving response times (TechRepublic).

4. **Optimizing Operations**: Businesses are using AI to analyze complex data for various applications, such as farming

## What you can do with this

One AgensgraphVector store gives you, over the same nodes:

- **semantic search** (HNSW) with scores + metadata,
- **metadata-filtered** retrieval (dates, domains, ranges),
- **hybrid** vector+keyword retrieval (RRF, tunable),
- **recall tuning** via `search_options={"hnsw.ef_search": N}`,
- a drop-in LangChain **retriever** for any LCEL RAG chain or agent.

```python
chain.invoke("your question here")
vector.similarity_search("a topic", k=8, filter={"date": {"$gte": "2018-01-01"}})
```

Close the shared pool when done: `agens.close()`